# 03 · Create YOUR OWN Genie space and ask in plain English
**Session 3 · step 3 of 4 · ~25 min · the main hands-on**

🧠 **The idea.** A **Genie space** lets a business user ask questions in plain English and
get governed SQL + answers back — *without writing SQL.* But a good Genie space isn't
automatic: you choose the right tables, add instructions and example questions, and curate
it. In this module **you build your own** and feel what makes it answer well vs. poorly.

🏢 **Why FHLB-Topeka cares.** The day's working assumption is *foundation-building before
broad Genie exposure.* Building a space yourself — on governed data, scoped and curated — is
exactly that foundation: you learn the pattern before you hand it to the business.

In [ ]:
# ============================================================
#  WORKSHOP CONFIG (1 of 2)  —  pick your catalog, THEN run the next cell
# ============================================================
# Mode A (live, our workspace):  serverless_stable_6fhczt_catalog  (the default)
# Mode B (portable / Free Edition): type the catalog 00_LOAD_DATA used
#   (the one you created, or an existing one you loaded into). Same value in every module.
# This cell only creates the picker at the top of the notebook.
dbutils.widgets.text("catalog", "serverless_stable_6fhczt_catalog", "Catalog")
print("↑ Set the 'Catalog' widget at the top of the notebook, then run the next cell.")

In [ ]:
# ---- WORKSHOP CONFIG (2 of 2)  —  apply the selected catalog ----
CATALOG = dbutils.widgets.get("catalog").strip()
assert CATALOG, "Set the 'Catalog' widget at the top of the notebook, then re-run this cell."

GOLD   = f"{CATALOG}.fhlb_gold"     # governed, analyst-ready data products (read-only)
SILVER = f"{CATALOG}.fhlb_silver"   # cleaned/typed layer (we use the HPI time series here)

spark.sql(f"USE CATALOG {CATALOG}")
print(f"Catalog: {CATALOG}  ·  gold: {GOLD}")

## Step 1 — Create the space (name it after yourself)
1. Left nav → **Genie** → **New** (or **Genie** from the SQL editor).
2. **Name it `firstname-lastname FHLB Advances`** (use *your* name — many of us share this
   workspace, so unique names keep spaces from colliding).
3. Pick your **SQL warehouse** when prompted.

## Step 2 — Add the right tables as assets
Add these governed gold tables (Genie answers only from what you add):
- `fhlb_gold.portfolio_concentration`
- `fhlb_gold.member_advance_summary`
- `fhlb_gold.member_collateral_capacity`
- `fhlb_gold.mpf_portfolio_summary`

**Why these four:** they carry the concentration, maturity, collateral, and credit story.
Leaving raw/irrelevant tables out is a *feature* — a tight space answers better.

## Step 3 — Ask, in plain English
Open the space full-screen (its own tab) and ask these. Compare each answer to what you
found in SQL (module 02) — **the numbers should match.**

1. *"Which members have the largest outstanding advances?"*
2. *"What share of the total advance book does the top member hold?"*
3. *"Which members are undercollateralized?"*
4. *"What is the MPF delinquency rate by product?"*

👀 **Watch for:** does Genie pick the right table? Does its number match your SQL? When it's
off, that's your cue to **curate** (next step).

## Step 4 — Curate: make it answer better
This is the skill. In the space's settings:
- **General instructions** — add business context, e.g.
  *"Advances are outstanding par in USD. 'Concentration' = a member's share_of_advance_book.
  A member is undercollateralized when collateral_utilization_pct > 1.0. Numbers are
  illustrative, not production."*
- **Example / trusted questions (SQL snippets)** — save your module-02 queries as trusted
  answers (e.g. the top-5 concentration query). Genie reuses them and generalizes.
- **Column descriptions & synonyms** — teach it that "the book" = the advance book, "delinquent"
  maps to `delinquency_rate`.

Re-ask question 2 ("what share does the top member hold?") after adding instructions — it
should now reliably return **~24.3% (Midwest Savings Bank).**

## Step 5 — (Optional) verify a Genie answer against SQL, right here
Genie should agree with governed SQL. If you want to prove it, run the matching query in a
cell and eyeball it against Genie's answer.

In [ ]:
%sql
-- The "top member's share" ground truth (should match your Genie answer: ~24.3%)
SELECT member_name, ROUND(share_of_advance_book*100,1) AS pct_of_book
FROM fhlb_gold.portfolio_concentration
ORDER BY share_of_advance_book DESC
LIMIT 1

## 🧑‍💻 Your Turn
- Add **one general instruction** and **one trusted question**, then ask a question you *didn't*
  seed and see if curation helped it generalize.
- Ask a deliberately **ambiguous** question ("who's risky?") and watch how Genie interprets it —
  then tighten your instructions so it picks a definition.

## ⚠️ Fallback / governance notes
- If Genie returns a number that doesn't match SQL, trust the SQL and curate the space —
  that gap *is* the lesson.
- **Sharing is governed:** a Genie space respects Unity Catalog permissions — a viewer only
  ever sees data they're already entitled to. That's why "foundation before broad exposure"
  works: the governance is already underneath.

## 🌟 Optional
Create a **second** space on housing (`housing_market_reference` + `fhlb_silver.fhfa_hpi`) and
ask *"how has district housing changed since 2008?"* — compare curated vs. uncurated answers.

---
### ✅ Done
**Next:** open [`04_dashboard`]($./04_dashboard) — turn these answers into a shareable view.